# Budget Reallocation Scenario

Loads the fitted model directly from disk via `MMM.load()` — self-contained,
no dependency on 02_mmm_model.ipynb's kernel still being alive.

In [1]:
import pandas as pd
import numpy as np
from pymc_marketing.mmm import MMM

df = pd.read_csv('../data/raw/mmm_weekly_data.csv', parse_dates=['date'])
channels = ['tv_spend', 'paid_search_spend', 'paid_social_spend', 'display_spend', 'promo_spend']
df['time_trend'] = np.arange(len(df)) / len(df)

# Reloads the fitted model, including its posterior, without refitting
mmm = MMM.load('../data/processed/mmm_fit.nc')
target_scale = mmm.idata.constant_data['target_scale'].item()
print("Loaded fitted model.")

g++ not available, if using conda: `conda install gxx`


Loaded fitted model.


In [2]:
X_baseline = df[['date'] + channels + ['price_index', 'time_trend']]

df_scenario = df.copy()
shift_amount = df['display_spend'] * 0.30
df_scenario['display_spend'] = df['display_spend'] - shift_amount
df_scenario['paid_search_spend'] = df['paid_search_spend'] + shift_amount * 0.6
df_scenario['promo_spend'] = df['promo_spend'] + shift_amount * 0.4

X_scenario = df_scenario[['date'] + channels + ['price_index', 'time_trend']]

In [3]:
pred_baseline = mmm.sample_posterior_predictive(X_baseline, extend_idata=False)
pred_scenario = mmm.sample_posterior_predictive(X_scenario, extend_idata=False)

baseline_total = pred_baseline['y'].mean(dim='sample').sum().item() * target_scale
scenario_total = pred_scenario['y'].mean(dim='sample').sum().item() * target_scale

print(f"Baseline predicted total sales: {baseline_total:,.0f}")
print(f"Scenario predicted total sales: {scenario_total:,.0f}")
print(f"Projected lift: {scenario_total - baseline_total:,.0f} ({(scenario_total/baseline_total - 1)*100:.2f}%)")

Sampling: [y]


Output()

Sampling: [y]


Output()

Baseline predicted total sales: 4,484,596
Scenario predicted total sales: 4,505,876
Projected lift: 21,280 (0.47%)


In [4]:
shifted_dollars = shift_amount.sum()
lift_dollars = scenario_total - baseline_total
roi_of_shift = lift_dollars / shifted_dollars

print(f"Total dollars shifted: R${shifted_dollars:,.0f}")
print(f"Projected sales lift: R${lift_dollars:,.0f}")
print(f"Return per dollar reallocated: R${roi_of_shift:.2f}")

scenario_results = pd.DataFrame({
    'metric': ['baseline_total_sales', 'scenario_total_sales', 'projected_lift', 'lift_pct', 'dollars_shifted', 'return_per_dollar'],
    'value': [baseline_total, scenario_total, lift_dollars, (scenario_total/baseline_total - 1)*100, shifted_dollars, roi_of_shift]
})
scenario_results.to_csv('../data/processed/budget_reallocation_scenario.csv', index=False)
scenario_results

Total dollars shifted: R$92,493
Projected sales lift: R$21,280
Return per dollar reallocated: R$0.23


,metric,value
0,baseline_total_sales,4.484596e+06
1,scenario_total_sales,4.505876e+06
2,projected_lift,2.127957e+04
3,lift_pct,4.745036e-01
4,dollars_shifted,9.249255e+04
5,return_per_dollar,2.300680e-01
